# Load TSpec Data

In [ ]:
import os
import re
from bs4 import BeautifulSoup
from markdown import markdown
from app.utils.settings import DATA_DIRECTORY, CHUNKS_FILE
from app.utils.chunking import save_chunks

# This function loads markdown files, preserves formatting, and converts HTML tables to readable markdown/plaintext.
def load_tspec_data(directory):
    """
    Load .md files while preserving formatting (text and tables) in a readable form.
    For each .md file (filtered to Rel-18 / 28_series like the previous loader),
    returns a dict with:
      - release, series, spec (filename without .md)
      - original_md: original markdown text
      - processed_text: extracted text with tables converted to markdown/plaintext
      - html: HTML generated from the markdown (useful for debug/visualization)
    """
    data = []

    for release in os.listdir(directory):
        release_path = os.path.join(directory, release)
        if os.path.isdir(release_path) and release == "Rel-18":
            for series in os.listdir(release_path):
                series_path = os.path.join(release_path, series)
                if os.path.isdir(series_path) and series == "28_series":
                    for file in os.listdir(series_path):
                        if file.endswith('.md'):
                            file_path = os.path.join(series_path, file)
                            with open(file_path, 'r', encoding='utf-8') as f:
                                content = f.read()
                                spec_name = file[:-3]
                                
                                # Apply the same filter as the previous loader
                                if release == "Rel-18" and series == "28_series" and spec_name[:5] == "28532":

                                    # Convert markdown to HTML (support tables and fenced code)
                                    html = markdown(content, extensions=["tables", "fenced_code", "codehilite"])

                                    # Use BeautifulSoup to find HTML tables and convert them to readable markdown/plaintext
                                    soup = BeautifulSoup(html, "html.parser")

                                    # Extract text from the resulting HTML, preserving line breaks
                                    processed_text = soup.get_text(separator="\n", strip=True)

                                    # Normalize multiple blank lines
                                    processed_text = re.sub(r"\n\s*\n+", "\n\n", processed_text).strip()

                                    data.append({
                                        "release": release,
                                        "series": series,
                                        "spec": spec_name,
                                        "original_md": content,
                                        "processed_text": processed_text,
                                        "html": str(soup)
                                    })
    return data

In [ ]:
# directory_path = '../../../../../Dataset/TSpec-LLM/3GPP-clean'
directory_path = DATA_DIRECTORY
tspec_data = load_tspec_data(directory_path)

In [ ]:
print(f"Sample document:\n{tspec_data[0]['original_md'][:10000]}")

In [ ]:
# print(f"Total documents loaded: {len(tspec_data)}")
# print(f"Sample document: {tspec_data[0]['processed_text']}")
print(f"Sample document: {tspec_data[0]['html']}")


# Build chunks (isolate Tables)

## Separate Text and Tables

In [ ]:
from typing import List, Dict, Any
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
def is_table_line(line: str) -> bool:
    """
    Heuristic to detect if a line belongs to a 3GPP-style Markdown/ASCII table.
    """
    stripped = line.strip()
    if not stripped:
        return False
    # Classic separator: +---+---+ or similar
    if re.match(r'^\+[-+=| ]+\+$', stripped):
        return True
    # Row with | separators, starting or ending with |
    if '|' in stripped and (stripped.startswith('|') or stripped.endswith('|')):
        return True
    return False

def extract_tables_pure_md(md_content: str, release: str = "unknown", series: str = "unknown", spec: str = "unknown") -> tuple[str, List[Dict[str, str]]]:
    """
    Extract complete table blocks from raw Markdown text.
    Inserts a numbered placeholder in clean_text where each table was removed, with metadata.
    Returns:
        - clean_text: markdown without tables, but with placeholders like "[Extracted Table: 1 from spec (Release Y, Series Z)]"
        - tables: list of dicts with {'number': '1', 'content': full table markdown} (for easy numbering in chunks)
    """
    lines = md_content.splitlines(keepends=False)
    clean_lines: List[str] = []
    tables: List[Dict[str, str]] = []
    current_table: List[str] = []
    table_counter = 1  # Simple counter for table numbers (per document)
    i = 0

    while i < len(lines):
        line = lines[i].rstrip("\r\n")

        if is_table_line(line):
            current_table.append(line)
            i += 1
            # Continue collecting while it looks like table
            while i < len(lines) and is_table_line(lines[i].rstrip("\r\n")):
                current_table.append(lines[i].rstrip("\r\n"))
                i += 1

            # Save only if it looks like a real table (min 3 lines)
            if len(current_table) >= 3:
                table_md = "\n".join(current_table)
                tables.append({
                    'number': str(table_counter),
                    'content': table_md
                })

                # Insert placeholder in clean text (with number and metadata)
                placeholder = f"[Extracted Table: {table_counter} from {spec} (Release {release}, Series {series})]"
                clean_lines.append(placeholder)
                table_counter += 1
            else:
                # False positive → return to clean text
                clean_lines.extend(current_table)
            current_table = []
        else:
            clean_lines.append(line)
            i += 1

    clean_text = "\n".join(clean_lines)
    # Normalize excessive newlines
    clean_text = re.sub(r'\n{3,}', '\n\n', clean_text).strip()

    return clean_text, tables

In [ ]:
# Run the extraction function
clean_text, tables = extract_tables_pure_md(tspec_data[0]['original_md'])

In [ ]:
# Print results for debug
print("=== Clean Text (without tables) ===\n")
print(clean_text[:10000] + "..." if len(clean_text) > 500 else clean_text)  # Show first 500 chars

In [ ]:
print("\n=== Extracted Tables ===\n")
if tables:
    for table_dict in tables:
        print(f"Table Number: {table_dict['number']}")
        print(f"Content:\n{table_dict['content']}\n")
else:
    print("No tables found in sample.")

## Split Text and create chunks

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
    (r"\n\d+(\.\d+)*\s", "Section"),  # Matches 1 Scope, 4.3.1 Any, etc.
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # Keep header inside chunk for context
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,       # Good for 3GPP: fits most sections/tables
    chunk_overlap=300,     # Overlap for cross-section references
    separators=[
        "\n\n\n",
        r"\n\d+(\.\d+)*\s+[A-Za-z]",  # Prioritize numbered sections like "4.3.1 Definition"
        "\n# ", "\n## ", "\n### ",
        "\n\n", "\n", " ", ""
    ],
)

In [ ]:
def divide_into_chunks(tspec_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Process loaded data into chunks.
    - Extracts tables as intact chunks first (with numbered placeholders in clean_text for coherence)
    - Splits remaining text with markdown_splitter + recursive_splitter
    - Adds metadata for RAG (release, series, spec, chunk_index, is_table, headers)
    """
    dataset_chunks = []
    chunk_index = 0

    for document in tspec_data:
        release = document['release']
        series = document['series']
        spec = document['spec']
        original_md = document['original_md']  # Use raw MD for accurate table extraction

        # Extract tables with placeholders and numbering (pass metadata)
        clean_text, table_blocks = extract_tables_pure_md(original_md, release=release, series=series, spec=spec)

        # Split the clean text (now with numbered placeholders)
        header_chunks = markdown_splitter.split_text(clean_text)

        # Refine with recursive splitter
        for header_chunk in header_chunks:
            char_chunks = recursive_splitter.split_text(header_chunk.page_content)
            for chunk_text in char_chunks:
                cleaned_chunk = chunk_text.strip()
                if cleaned_chunk:  # Skip empty chunks
                    dataset_chunks.append({
                        'release': release,
                        'series': series,
                        'spec': spec,
                        'content': cleaned_chunk,
                        'chunk_index': chunk_index,
                        'is_table': False,
                        'headers': header_chunk.metadata  # e.g., {'Section': '4.3.1'}
                    })
                    chunk_index += 1

        # Add each extracted table as a separate chunk (with number in title)
        for table_dict in table_blocks:
            table_number = table_dict['number']
            table_content = table_dict['content']
            enriched_table = f"**Spec {spec}, Release {release}, Series {series}**\nTable {table_number}\n\n{table_content}"
            dataset_chunks.append({
                'release': release,
                'series': series,
                'spec': spec,
                'content': enriched_table,
                'chunk_index': chunk_index,
                'is_table': True,
                'headers': {'Table': f'Full extracted table {table_number}'}
            })
            chunk_index += 1

    print(f"Generated {len(dataset_chunks)} chunks ({sum(1 for c in dataset_chunks if c['is_table'])} tables)")
    return dataset_chunks

In [ ]:
tspec_chunks = divide_into_chunks(tspec_data)

In [ ]:
# Check the result
print(f"Total chunks created: {len(tspec_chunks)}")
print(f"Example chunk:\n {tspec_chunks[100]}")
print(f"Example table chunk:\n {tspec_chunks[700]}")

In [ ]:
chunks_path = CHUNKS_FILE
save_chunks(tspec_chunks, chunks_path)

# Build chunks (without isolate tables)

In [ ]:
# from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
# import re

# # Configure headers for splitting
# headers_to_split_on = [
#     ("#", "Header 1"),
#     ("##", "Header 2"),
#     ("###", "Header 3"),
#     ("####", "Header 4"),
#     (r"\n\d+(\.\d+)*\s", "Section"),  # Matches 1 Scope, 4.3.1 Any, etc.
# ]

# # Initialize the MarkdownHeaderTextSplitter
# markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

# # Configure the RecursiveCharacterTextSplitter
# chunk_size = 1200
# chunk_overlap = 300
# separators=[
#         "\n\n\n",  # Big blocks (ex: after Contents)
#         r"\n\d+(\.\d+)*\s+[A-Za-z]",  # Prioritize numbered sections like "4.3.1 Definition"
#         "\n# ", "\n## ", "\n### ",
#         "\n\n", "\n", " ", ""
#     ]
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=chunk_size,
#     chunk_overlap=chunk_overlap,
#     separators=separators
# )

# # Function to divide content into chunks
# def divide_into_chunks(tspec_data):
#     dataset_chunks = []

#     for document in tspec_data:
#         release = document['release']
#         series = document['series']
#         spec = document['spec']
#         content = document['original_md']
        
#         # Split by Markdown headers
#         header_chunks = markdown_splitter.split_text(content)
        
#         # Further split the chunks by characters
#         for header_chunk in header_chunks:
#             char_chunks = text_splitter.split_text(header_chunk.page_content)
#             for chunk in char_chunks:
#                 dataset_chunks.append({
#                     'release': release,
#                     'series': series,
#                     'spec': spec,
#                     'content': chunk
#                 })

#     return dataset_chunks

In [ ]:
# tspec_chunks = divide_into_chunks(tspec_data)

In [ ]:
# # Check the result
# print(f"Total chunks created: {len(tspec_chunks)}")
# print(f"Example chunk:\n {tspec_chunks[10]}")

In [ ]:
# chunks_path = CHUNKS_FILE
# save_chunks(tspec_chunks, chunks_path)